In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

import joblib

In [ ]:
df = pd.read_csv('/content/WA_Fn-UseC_-HR-Employee-Attrition.csv')

df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df['Attrition'].value_counts()

In [ ]:
sns.countplot(data=df, x='Attrition')
plt.title('Employee Attrition Distribution')
plt.show()

In [ ]:
sns.countplot(data=df, x='OverTime', hue='Attrition')
plt.title('Overtime vs Employee Attrition')
plt.show()

In [ ]:
sns.boxplot(data=df, x='Attrition', y='Age')
plt.title('Age vs Employee Attrition')
plt.show()

In [ ]:
df.drop(
    ['EmployeeCount', 'EmployeeNumber', 'Over18', 'StandardHours'],
    axis=1,
    inplace=True
)

df.shape

In [ ]:
X = df.drop('Attrition', axis=1)

y = df['Attrition']

In [ ]:
print("Features:", X.shape)
print("Target:", y.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

In [ ]:
numerical_features = X_train.select_dtypes(
    include=['int64', 'float64']
).columns

categorical_features = X_train.select_dtypes(
    include=['object']
).columns

print("Numerical features:", len(numerical_features))
print("Categorical features:", len(categorical_features))

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)

In [ ]:
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

In [ ]:
print(X_train_processed.shape)
print(X_test_processed.shape)

In [ ]:
le = LabelEncoder()

y_train_encoded = le.fit_transform(y_train)

y_test_encoded = le.transform(y_test)

print(le.classes_)

In [ ]:
def evaluate_model(y_true, y_pred):

    print("Accuracy:", accuracy_score(y_true, y_pred))

    print("Precision:", precision_score(y_true, y_pred, zero_division=0))

    print("Recall:", recall_score(y_true, y_pred, zero_division=0))

    print("F1 Score:", f1_score(y_true, y_pred, zero_division=0))

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred))

In [ ]:
lr_model = LogisticRegression(max_iter=1000)

lr_model.fit(X_train_processed, y_train_encoded)

lr_pred = lr_model.predict(X_test_processed)

evaluate_model(y_test_encoded, lr_pred)

In [ ]:
dt_model = DecisionTreeClassifier(random_state=42)

dt_model.fit(X_train_processed, y_train_encoded)

dt_pred = dt_model.predict(X_test_processed)

evaluate_model(y_test_encoded, dt_pred)

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train_processed, y_train_encoded)

rf_pred = rf_model.predict(X_test_processed)

evaluate_model(y_test_encoded, rf_pred)

In [ ]:
lr_balanced = LogisticRegression(
    max_iter=1000,
    class_weight='balanced'
)

lr_balanced.fit(X_train_processed, y_train_encoded)

lr_balanced_pred = lr_balanced.predict(X_test_processed)

evaluate_model(y_test_encoded, lr_balanced_pred)

In [ ]:
dt_balanced = DecisionTreeClassifier(
    random_state=42,
    class_weight='balanced'
)

dt_balanced.fit(X_train_processed, y_train_encoded)

dt_balanced_pred = dt_balanced.predict(X_test_processed)

evaluate_model(y_test_encoded, dt_balanced_pred)

In [ ]:
rf_balanced = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight='balanced'
)

rf_balanced.fit(X_train_processed, y_train_encoded)

rf_balanced_pred = rf_balanced.predict(X_test_processed)

evaluate_model(y_test_encoded, rf_balanced_pred)

In [ ]:
results = pd.DataFrame({
    'Model': [
        'Logistic Regression',
        'Decision Tree',
        'Random Forest',
        'Balanced Logistic Regression',
        'Balanced Decision Tree',
        'Balanced Random Forest',
    ],

    'Accuracy': [
        accuracy_score(y_test_encoded, lr_pred),
        accuracy_score(y_test_encoded, dt_pred),
        accuracy_score(y_test_encoded, rf_pred),
        accuracy_score(y_test_encoded, lr_balanced_pred),
        accuracy_score(y_test_encoded, dt_balanced_pred),
        accuracy_score(y_test_encoded, rf_balanced_pred),
    ],

    'Precision': [
        precision_score(y_test_encoded, lr_pred, zero_division=0),
        precision_score(y_test_encoded, dt_pred, zero_division=0),
        precision_score(y_test_encoded, rf_pred, zero_division=0),
        precision_score(y_test_encoded, lr_balanced_pred, zero_division=0),
        precision_score(y_test_encoded, dt_balanced_pred, zero_division=0),
        precision_score(y_test_encoded, rf_balanced_pred, zero_division=0),
    ],

    'Recall': [
        recall_score(y_test_encoded, lr_pred),
        recall_score(y_test_encoded, dt_pred),
        recall_score(y_test_encoded, rf_pred),
        recall_score(y_test_encoded, lr_balanced_pred),
        recall_score(y_test_encoded, dt_balanced_pred),
        recall_score(y_test_encoded, rf_balanced_pred),
    ],

    'F1 Score': [
        f1_score(y_test_encoded, lr_pred),
        f1_score(y_test_encoded, dt_pred),
        f1_score(y_test_encoded, rf_pred),
        f1_score(y_test_encoded, lr_balanced_pred),
        f1_score(y_test_encoded, dt_balanced_pred),
        f1_score(y_test_encoded, rf_balanced_pred),
    ]
})

results.round(4)

In [ ]:
final_model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced'
)

final_model.fit(X_train_processed, y_train_encoded)

In [ ]:
final_pred = final_model.predict(X_test_processed)

evaluate_model(y_test_encoded, final_pred)

In [ ]:
print(
    classification_report(
        y_test_encoded,
        final_pred,
        target_names=le.classes_
    )
)

In [ ]:
joblib.dump(
    {
        'model': final_model,
        'preprocessor': preprocessor,
        'label_encoder': le,

        'feature_columns': list(X.columns),

        'categorical_features': list(categorical_features),

        'numerical_features': list(numerical_features),

        'category_options': {
            col: df[col].dropna().unique().tolist()
            for col in categorical_features
        },

        'numeric_defaults': X_train[numerical_features].median().to_dict()
    },

    '/content/employee_attrition_model.pkl'
)

print("Model saved successfully!")

In [ ]:
import os

print(
    "Model exists:",
    os.path.exists('/content/employee_attrition_model.pkl')
)

In [ ]:
!pip install streamlit

In [ ]:
import streamlit as st
import pandas as pd
import joblib

print("Streamlit and required libraries are ready!")

In [ ]:
!streamlit run /content/app.py --server.address 0.0.0.0 --server.port 8501

In [ ]:
import os

print("App file:", os.path.exists('/content/app.py'))
print("Model file:", os.path.exists('/content/employee_attrition_model.pkl'))

### Diagnose Streamlit Application Error
Run this cell to check the running background processes, capture recent outputs, and inspect if any model parameters or layout rendering issues are crashing the server.

In [ ]:
# 1. Check if the Streamlit process is actually running
!ps aux | grep streamlit

# 2. Run the application directly in foreground mode for 5 seconds to capture error tracebacks
import subprocess
import time

print("\n--- Launching Streamlit in check mode to capture error tracebacks ---")
process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8502", "--server.address", "0.0.0.0"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

time.sleep(6)
process.terminate()
stdout, stderr = process.communicate()

print("STDOUT Output:\n", stdout)
print("\nSTDERR Errors:\n", stderr)

### Alternative: Expose App via Localtunnel (Bypasses Iframe Security)
If the inline view or standard Colab proxy link remains blank due to browser security, this method generates a direct public URL.

### Alternative: Tunnel via Cloudflare (Highly Reliable Direct Link)
This method downloads the official Cloudflare tunnel client and safely forwards your Streamlit app (port 8501) to a secure public URL.

In [ ]:
# 1. Download and install cloudflared package
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

# 2. Start a tunnel forwarding port 8501 in the background and log output to a file
import subprocess
import time
import re

# Stop any existing cloudflared tunnels first
!pkill cloudflared

print("Starting Cloudflare Tunnel...")
with open("cf_tunnel.log", "w") as log_file:
    subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:8501"], stdout=log_file, stderr=log_file)

# 3. Wait a few seconds and extract the generated public URL
time.sleep(6)

with open("cf_tunnel.log", "r") as log_file:
    log_content = log_file.read()

# Search for the trycloudflare.com link in logs
urls = re.findall(r'https://[a-zA-Z0-9\-]+\.trycloudflare\.com', log_content)
if urls:
    print(f"\nSuccess! Your app is live at the secure URL below:\n")
    print(f"👉 {urls[0]} 👈\n")
else:
    print("Could not find tunnel URL. Showing latest logs below:\n")
    print(log_content[-500:])

In [ ]:
%%writefile /content/app.py
import streamlit as st
import pandas as pd
import joblib
import numpy as np
import re

# --------------------------------------------------
# 1. PAGE CONFIGURATION
# --------------------------------------------------
st.set_page_config(
    page_title="TalentRisk Premier | Interactive HR Analyst",
    page_icon="🧬",
    layout="wide"
)

# Advanced Modern Executive Styling with Premium Colors
st.markdown("""
    <style>
    @import url("https://fonts.googleapis.com/css2?family=Plus+Jakarta+Sans:wght@400;500;600;700;800&display=swap");

    /* Global font & background upgrade */
    * {
        font-family: "Plus Jakarta Sans", sans-serif;
    }

    .stApp {
        background-color: #f8fafc;
    }

    /* Sophisticated Header Banner with Vibrant Color Gradient */
    .header-container {
        background: linear-gradient(135deg, #1e1b4b 0%, #312e81 40%, #4338ca 100%);
        padding: 40px;
        border-radius: 16px;
        color: #ffffff;
        margin-bottom: 30px;
        box-shadow: 0 10px 25px -5px rgba(49, 46, 129, 0.15);
        border-left: 8px solid #6366f1;
    }
    .header-title {
        font-weight: 800;
        font-size: 2.8rem;
        margin: 0;
        letter-spacing: -1px;
    }
    .header-subtitle {
        font-size: 1.15rem;
        opacity: 0.90;
        margin-top: 10px;
        font-weight: 400;
    }

    /* Extreme High-Visibility Contrast Labels (Styled & Cleanly Spaced) */
    label, .stWidgetLabel, div[data-testid="stWidgetLabel"] p {
        font-size: 1.1rem !important;
        font-weight: 700 !important;
        color: #0f172a !important; /* Near black Slate-900 */
        letter-spacing: 0.2px;
        margin-bottom: 6px !important;
    }

    /* Beautiful custom section header labels with neon borders */
    .section-banner-blue {
        font-size: 1.25rem !important;
        font-weight: 800 !important;
        color: #1e3a8a !important;
        border-left: 5px solid #2563eb !important;
        padding-left: 12px !important;
        margin-top: 20px !important;
        margin-bottom: 20px !important;
        text-transform: uppercase;
        letter-spacing: 0.5px;
    }
    .section-banner-indigo {
        font-size: 1.25rem !important;
        font-weight: 800 !important;
        color: #312e81 !important;
        border-left: 5px solid #4f46e5 !important;
        padding-left: 12px !important;
        margin-top: 20px !important;
        margin-bottom: 20px !important;
        text-transform: uppercase;
        letter-spacing: 0.5px;
    }
    .section-banner-violet {
        font-size: 1.25rem !important;
        font-weight: 800 !important;
        color: #581c87 !important;
        border-left: 5px solid #7c3aed !important;
        padding-left: 12px !important;
        margin-top: 20px !important;
        margin-bottom: 20px !important;
        text-transform: uppercase;
        letter-spacing: 0.5px;
    }

    /* Custom subheaders for analysis outputs */
    .custom-subheader {
        font-size: 1.6rem !important;
        font-weight: 900 !important;
        color: #0f172a !important;
        margin-top: 25px !important;
        margin-bottom: 15px !important;
        border-left: 8px solid #4f46e5 !important;
        padding-left: 16px !important;
    }

    /* Highly polished card form containers */
    div[data-testid="stForm"] {
        background: #ffffff;
        padding: 35px !important;
        border-radius: 16px !important;
        border: 1px solid #cbd5e1 !important;
        box-shadow: 0 10px 25px -5px rgba(0, 0, 0, 0.05) !important;
    }

    /* High impact Metric formatting */
    div[data-testid="stMetricValue"] {
        font-size: 3rem !important;
        font-weight: 900 !important;
        color: #090d16 !important; /* Pitch black */
    }
    div[data-testid="stMetricDelta"] p {
        font-size: 1.3rem !important;
        font-weight: 900 !important;
    }

    /* Premium Custom Action Button */
    div.stButton > button {
        background: linear-gradient(90deg, #312e81 0%, #4338ca 100%) !important;
        color: #ffffff !important;
        font-size: 1.2rem !important;
        font-weight: 800 !important;
        border-radius: 10px !important;
        padding: 14px 28px !important;
        width: 100% !important;
        border: none !important;
        box-shadow: 0 4px 15px rgba(67, 56, 202, 0.25) !important;
        transition: all 0.2s ease;
    }
    div.stButton > button:hover {
        background: linear-gradient(90deg, #1e1b4b 0%, #312e81 100%) !important;
        transform: translateY(-1px);
        box-shadow: 0 6px 20px rgba(67, 56, 202, 0.35) !important;
    }
    </style>
""", unsafe_allow_html=True)

# --------------------------------------------------
# 2. LOAD SAVED MODEL
# --------------------------------------------------
model_data = joblib.load('/content/employee_attrition_model.pkl')

model = model_data['model']
preprocessor = model_data['preprocessor']
label_encoder = model_data['label_encoder']

feature_columns = model_data['feature_columns']
categorical_features = model_data['categorical_features']
numerical_features = model_data['numerical_features']

category_options = model_data['category_options']
numeric_defaults = model_data['numeric_defaults']

# --------------------------------------------------
# Helper to dynamically inject formatting spaces
# --------------------------------------------------
def clean_label(text):
    # Turn camelCase or PascalCase into spaced words
    spaced = re.sub(r'(?<!^)(?=[A-Z])', ' ', text)
    # Fix abbreviations
    spaced = spaced.replace("Curr ", "Current ").replace("Rate", " Rate").replace("  ", " ")
    return spaced.strip()

# --------------------------------------------------
# 3. NUMERICAL INPUT LIMITS
# --------------------------------------------------
numeric_limits = {
    'Age': (18, 60),
    'DailyRate': (102, 1499),
    'DistanceFromHome': (1, 29),
    'Education': (1, 5),
    'EnvironmentSatisfaction': (1, 4),
    'HourlyRate': (30, 100),
    'JobInvolvement': (1, 4),
    'JobLevel': (1, 5),
    'JobSatisfaction': (1, 4),
    'MonthlyIncome': (1009, 19999),
    'MonthlyRate': (2094, 26999),
    'NumCompaniesWorked': (0, 9),
    'PercentSalaryHike': (11, 25),
    'PerformanceRating': (3, 4),
    'RelationshipSatisfaction': (1, 4),
    'StockOptionLevel': (0, 3),
    'TotalWorkingYears': (0, 40),
    'TrainingTimesLastYear': (0, 6),
    'WorkLifeBalance': (1, 4),
    'YearsAtCompany': (0, 40),
    'YearsInCurrentRole': (0, 18),
    'YearsSinceLastPromotion': (0, 15),
    'YearsWithCurrManager': (0, 17)
}

def run_prediction(input_data):
    input_df = pd.DataFrame([input_data], columns=feature_columns)
    processed_data = preprocessor.transform(input_df)
    prediction = model.predict(processed_data)
    predicted_label = label_encoder.inverse_transform(prediction)[0]

    probabilities = model.predict_proba(processed_data)[0]
    yes_index = list(model.classes_).index(label_encoder.transform(["Yes"])[0])
    leave_probability = probabilities[yes_index] * 100
    return predicted_label, leave_probability

# --------------------------------------------------
# 4. APP TITLE LAYOUT
# --------------------------------------------------
st.markdown("""
    <div class="header-container">
        <div class="header-title">🧬 TalentRisk AI Dashboard</div>
        <div class="header-subtitle">An executive intelligence workspace engineered to decode personnel attrition markers and simulate strategic retention models.</div>
    </div>
""", unsafe_allow_html=True)

tab1, tab2 = st.tabs(["📋 Employee Profiler Workspace", "⚡ Interactive Retention Simulator"])

if 'base_inputs' not in st.session_state:
    st.session_state.base_inputs = {}

# --------------------------------------------------
# TAB 1: EMPLOYEE PROFILER WORKSPACE
# --------------------------------------------------
with tab1:
    st.markdown("<p style='font-size: 1.15rem; font-weight: 500; color: #475569; margin-bottom: 20px;'>Complete the analytical profile form below to compute the initial attrition risk baseline.</p>", unsafe_allow_html=True)

    with st.form("employee_form", clear_on_submit=False):
        form_inputs = {}

        st.markdown('<div class="section-banner-blue">🏢 Department & Organizational Role Profile</div>', unsafe_allow_html=True)
        col_a1, col_a2, col_a3 = st.columns(3)
        role_features = ['Department', 'JobRole', 'JobLevel', 'JobInvolvement', 'BusinessTravel', 'OverTime']
        for i, feat in enumerate(role_features):
            label_display = clean_label(feat)
            with (col_a1 if i % 3 == 0 else col_a2 if i % 3 == 1 else col_a3):
                if feat in categorical_features:
                    form_inputs[feat] = st.selectbox(label_display, category_options[feat], index=0)
                else:
                    min_v, max_v = numeric_limits[feat]
                    form_inputs[feat] = st.number_input(label_display, min_v, max_v, int(numeric_defaults[feat]))

        st.markdown('<div class="section-banner-indigo">💸 Compensation Structure & Demographics</div>', unsafe_allow_html=True)
        col_b1, col_b2, col_b3 = st.columns(3)
        comp_features = ['MonthlyIncome', 'PercentSalaryHike', 'StockOptionLevel', 'Age', 'Gender', 'MaritalStatus']
        for i, feat in enumerate(comp_features):
            label_display = clean_label(feat)
            with (col_b1 if i % 3 == 0 else col_b2 if i % 3 == 1 else col_b3):
                if feat in categorical_features:
                    form_inputs[feat] = st.selectbox(label_display, category_options[feat], index=0)
                else:
                    min_v, max_v = numeric_limits[feat]
                    form_inputs[feat] = st.number_input(label_display, min_v, max_v, int(numeric_defaults[feat]))

        st.markdown('<div class="section-banner-violet">🌱 Historic Tenure & Work Sentiment Factors</div>', unsafe_allow_html=True)
        col_c1, col_c2, col_c3 = st.columns(3)
        remaining_features = [f for f in feature_columns if f not in role_features + comp_features]
        for i, feat in enumerate(remaining_features):
            label_display = clean_label(feat)
            with (col_c1 if i % 3 == 0 else col_c2 if i % 3 == 1 else col_c3):
                if feat in categorical_features:
                    form_inputs[feat] = st.selectbox(label_display, category_options[feat], index=0)
                else:
                    min_v, max_v = numeric_limits[feat]
                    form_inputs[feat] = st.number_input(label_display, min_v, max_v, int(numeric_defaults[feat]))

        st.markdown("<br>", unsafe_allow_html=True)
        submitted = st.form_submit_button("🚀 Calculate Executive Risk Check")

    if submitted:
        st.session_state.base_inputs = form_inputs

    if st.session_state.base_inputs:
        base_label, base_prob = run_prediction(st.session_state.base_inputs)

        st.markdown("<br>", unsafe_allow_html=True)
        st.markdown('<div class="custom-subheader">Baseline Assessment Result</div>', unsafe_allow_html=True)
        col_res1, col_res2 = st.columns([1.2, 2])

        with col_res1:
            if base_label == "Yes":
                st.error("⚠️ Attrition Risk Flagged: High exit indicators present.")
            else:
                st.success("✅ Profile Stable: This employee is highly likely to stay.")

        with col_res2:
            st.metric("Baseline Retention Risk", f" {base_prob:.1f}%")
            st.progress(float(base_prob / 100))

# --------------------------------------------------
# TAB 2: RETENTION MITIGATION SIMULATOR
# --------------------------------------------------
with tab2:
    st.markdown('<div class="custom-subheader">Mitigation Adjuster Workspace</div>', unsafe_allow_html=True)
    st.markdown("<p style='font-size: 1.2rem; font-weight: 600; color: #334155;'>Simulate key strategic alterations to instantly calculate risk impact comparisons.</p>", unsafe_allow_html=True)

    if not st.session_state.base_inputs:
        st.info("💡 To start simulation planning, please configure a baseline profile first inside 'Baseline Profiler Workspace'.")
    else:
        scenario_inputs = st.session_state.base_inputs.copy()
        baseline_label, baseline_prob = run_prediction(st.session_state.base_inputs)

        col_ctrl1, col_ctrl2, col_ctrl3 = st.columns(3)

        with col_ctrl1:
            st.markdown("<div style='font-size: 1.25rem; font-weight: 800; color: #1e3a8a; margin-bottom: 12px;'>💳 Compensation Levers</div>", unsafe_allow_html=True)
            scenario_inputs['MonthlyIncome'] = st.slider(
                "Modify Monthly Income ($)",
                min_value=numeric_limits['MonthlyIncome'][0],
                max_value=numeric_limits['MonthlyIncome'][1],
                value=int(scenario_inputs['MonthlyIncome']),
                step=100
            )
            scenario_inputs['PercentSalaryHike'] = st.slider(
                "Modify Salary Hike Percentage (%)",
                min_value=numeric_limits['PercentSalaryHike'][0],
                max_value=numeric_limits['PercentSalaryHike'][1],
                value=int(scenario_inputs['PercentSalaryHike'])
            )

        with col_ctrl2:
            st.markdown("<div style='font-size: 1.25rem; font-weight: 800; color: #1e3a8a; margin-bottom: 12px;'>⚖️ Schedule & Work Balance Levers</div>", unsafe_allow_html=True)
            ot_index = category_options['OverTime'].index(scenario_inputs['OverTime'])
            scenario_inputs['OverTime'] = st.selectbox(
                "Overtime Policy Status",
                category_options['OverTime'],
                index=ot_index
            )
            scenario_inputs['WorkLifeBalance'] = st.slider(
                "Adjust Work Life Balance Index",
                min_value=numeric_limits['WorkLifeBalance'][0],
                max_value=numeric_limits['WorkLifeBalance'][1],
                value=int(scenario_inputs['WorkLifeBalance'])
            )

        with col_ctrl3:
            st.markdown("<div style='font-size: 1.25rem; font-weight: 800; color: #1e3a8a; margin-bottom: 12px;'>🌱 Workplace Satisfaction Levers</div>", unsafe_allow_html=True)
            scenario_inputs['JobSatisfaction'] = st.slider(
                "Adjust Job Satisfaction Index",
                min_value=numeric_limits['JobSatisfaction'][0],
                max_value=numeric_limits['JobSatisfaction'][1],
                value=int(scenario_inputs['JobSatisfaction'])
            )
            scenario_inputs['EnvironmentSatisfaction'] = st.slider(
                "Adjust Environment Satisfaction Index",
                min_value=numeric_limits['EnvironmentSatisfaction'][0],
                max_value=numeric_limits['EnvironmentSatisfaction'][1],
                value=int(scenario_inputs['EnvironmentSatisfaction'])
            )

        # Re-predict
        new_label, new_prob = run_prediction(scenario_inputs)
        delta_val = new_prob - baseline_prob

        st.markdown("--- ")
        st.markdown('<div class="custom-subheader">Live Scenario Performance Evaluation</div>', unsafe_allow_html=True)

        col_m1, col_m2, col_m3 = st.columns(3)
        with col_m1:
            st.metric(label="Baseline Risk Index", value=f"{baseline_prob:.1f}%")
        with col_m2:
            st.metric(
                label="Simulated Risk Index",
                value=f"{new_prob:.1f}%",
                delta=f"{delta_val:+.1f}%",
                delta_color="inverse"
            )
        with col_m3:
            st.write("**Simulation Strategy Outcome**")
            if new_label == "Yes":
                st.error("Action Recommended")
            else:
                st.success("Retention Plan Achieved!")

        st.progress(float(new_prob / 100))

        if new_prob < baseline_prob:
            st.success(f"Optimal Strategy Formulated! The customized changes successfully reduced overall risk probability by **{abs(delta_val):.1f}%**.")
        elif new_prob > baseline_prob:
            st.warning("Mitigation warning: This setup results in an unfavorable increase in predicted attrition risk.")


In [ ]:
# Gracefully restart Streamlit and rebuild Cloudflare Tunnel to map new features immediately
import subprocess
import time
import re

print("Gracefully restarting background Streamlit web instance...")
!pkill streamlit
time.sleep(2)
subprocess.Popen(["streamlit", "run", "/content/app.py", "--server.port", "8501", "--server.address", "0.0.0.0"])
time.sleep(3)

print("Reconnecting secure Cloudflare network mapping tunnel...")
!pkill cloudflared
time.sleep(2)

with open("cf_tunnel.log", "w") as log_file:
    subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:8501"], stdout=log_file, stderr=log_file)

time.sleep(6)

with open("cf_tunnel.log", "r") as log_file:
    log_content = log_file.read()

urls = re.findall(r'https://[a-zA-Z0-9\-]+\.trycloudflare\.com', log_content)
if urls:
    print(f"\nSuccess! Your high-visibility, simplified app is active at:\n")
    print(f"👉 {urls[0]} 👈\n")
else:
    print("Tunnel generation pending. Latest output:\n")
    print(log_content[-500:])